In [1]:
from scipy import io
import numpy as np
dataslices_output='data_slices/output'
batch_size=1
num_harmonics=4 
audio_dir='../../assets/trainingdata/chords/'
sampleRate,audio=io.wavfile.read(audio_dir+'session_original.wav')
audio=audio.astype(np.float32)
_,audio_hi=io.wavfile.read(audio_dir+'session_eq high.wav')
audio_hi=audio_hi.astype(np.float32)
_,audio_mid=io.wavfile.read(audio_dir+'session_eq mid.wav')
audio_mid=audio_mid.astype(np.float32)
_,audio_dist=io.wavfile.read(audio_dir+'session_tape dist.wav')
audio_dist=audio_dist.astype(np.float32)
print(len(audio))
use_augmentations=True


12576000


In [ ]:
from scipy import io
from common import plot_heatmap
from common import OUTPUT_DIM_NOTES
midi_file='../../assets/trainingdata/chords/session_NNTranscription-1-t1.mid'
dataslices_output='data_slices/output'
dataslices_onsets='data_slices/onsets'
num_augmentations=3 # The extra augmentations correspond to the high, mid and tape dist wav files





import numpy as np
import pypianoroll
import pretty_midi

# User-defined sample rate


# Load MIDI file
midi = pretty_midi.PrettyMIDI(midi_file)

# Calculate time resolution (step length in seconds per sample)
step_length = 1 / sampleRate

# Convert to piano roll with the desired time resolution
# pypianoroll resolution parameter is in ticks per beat,
# but you want time in seconds per sample. Use get_piano_roll from pretty_midi:
total_duration = midi.get_end_time()
n_steps = int(total_duration * sampleRate)
piano_roll = midi.get_piano_roll(fs=sampleRate)  # shape: (128, n_steps)
piano_roll=piano_roll[:OUTPUT_DIM_NOTES,:]


lenpianorool=piano_roll.shape[1]
desired_length=len(audio)
if lenpianorool<desired_length:
    # Pad with zeros
    padding = desired_length - lenpianorool
    piano_roll = np.pad(piano_roll, ((0, 0), (0, padding)), mode='constant')
elif lenpianorool>desired_length:
    # Truncate
    piano_roll = piano_roll[:, :desired_length]

print('Piano roll length after padding/truncating:', piano_roll.shape[1])
print('Audio length:', len(audio))


piano_roll=np.where(piano_roll>0,1,0)
for t in range(piano_roll.shape[1]):
    notes=piano_roll[range(0,OUTPUT_DIM_NOTES),t].max()
    if notes==0:
        piano_roll[OUTPUT_DIM_NOTES-1,t]=1

print(piano_roll.shape)
plot_heatmap(piano_roll)

from common import frame_size
from common import save_data_slices,reshape_to_nn_output


def savenn_output(directory,nn_classes):   
    nn_output=reshape_to_nn_output(nn_classes)
    if use_augmentations:
        print('Using augmentations')
        nn_output_shape=nn_output.shape
        new_shape=((num_augmentations+1)*nn_output_shape[0],nn_output_shape[1])
        print('Extending nn_output with shape'+str(nn_output.shape),' to new shape '+str(new_shape)+' to include the augmentations')
        nn_output_aug=np.resize(nn_output,new_shape)
        save_data_slices(directory,nn_output_aug,batch_size) 
    else:
        save_data_slices(directory,nn_output,batch_size) 
        
    return nn_output

savenn_output(dataslices_output,piano_roll)
# nn_onsets=savenn_output(dataslices_onsets,onsets2d)
# The piano_roll array:
# x-axis: sample index (at samplerate, matches audio directly)
# y-axis: MIDI note (0-127)

In [2]:
import numpy as np
import seaborn as sns
from fretboard import FretBoard
from common import frame_size,reshape_to_nn_input
    
filter =FretBoard(20,sampleRate)
numfilters=filter.get_num_filters()
filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
filter.process(audio,filterbank_out)
nn_input=reshape_to_nn_input(filterbank_out)
from common import save_data_slices
output_dir_input = 'data_slices/input'
save_data_slices(output_dir_input,nn_input,batch_size)

if use_augmentations:
    print('Processing Augmentations')
    #Audio hi
    filter =FretBoard(20,sampleRate)
    numfilters=filter.get_num_filters()
    filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
    filter.process(audio_hi,filterbank_out)
    nn_input=reshape_to_nn_input(filterbank_out)
    from common import save_data_slices
    output_dir_input = 'data_slices/input'
    save_data_slices(output_dir_input,nn_input,batch_size,len(audio))

    #Audio mid
    filter =FretBoard(20,sampleRate)
    numfilters=filter.get_num_filters()
    filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
    filter.process(audio_mid,filterbank_out)
    nn_input=reshape_to_nn_input(filterbank_out)
    from common import save_data_slices
    output_dir_input = 'data_slices/input'
    save_data_slices(output_dir_input,nn_input,batch_size,2*len(audio))

    #Audio dist
    filter =FretBoard(20,sampleRate)
    numfilters=filter.get_num_filters()
    filterbank_out=np.zeros((numfilters,len(audio)),dtype=np.float32)
    filter.process(audio_dist,filterbank_out)
    nn_input=reshape_to_nn_input(filterbank_out)
    from common import save_data_slices
    output_dir_input = 'data_slices/input'
    save_data_slices(output_dir_input,nn_input,batch_size,3*len(audio))


# if use_augmentations:
#     print('Processing Augmentations')
#     filter_hi =FretBoard(20,sampleRate)
#     filter_mid =FretBoard(20,sampleRate)
#     filter_dist =FretBoard(20,sampleRate)
#     print('num filters:'+str(numfilters)+' audio samples '+str(len(audio)))
    
    
    
#     filterbank_out_hi=np.zeros((numfilters,len(audio)),dtype=np.float32)
#     filterbank_out_mid=np.zeros((numfilters,len(audio)),dtype=np.float32)
#     filterbank_out_dist=np.zeros((numfilters,len(audio)),dtype=np.float32)
    
    
    
    
#     filter_hi.process(audio_hi,filterbank_out_hi)
#     filter_mid.process(audio_mid,filterbank_out_mid)
#     filter_dist.process(audio_dist,filterbank_out_dist)
# else: 
#     print('NOT Processing augmentations')

reshape data by a factor of 256...
(312, 12576000)
Reshaped the output data to  
(49125, 312)
Saving 49125 samples to disk with filenamuber offset 0...
Serialization complete. 49125 Files saved in 'data_slices/input'.
Processing Augmentations
reshape data by a factor of 256...
(312, 12576000)
Reshaped the output data to  
(49125, 312)
Saving 49125 samples to disk with filenamuber offset 49125...
Serialization complete. 49125 Files saved in 'data_slices/input'.
reshape data by a factor of 256...
(312, 12576000)
Reshaped the output data to  
(49125, 312)
Saving 49125 samples to disk with filenamuber offset 98250...
Serialization complete. 49125 Files saved in 'data_slices/input'.
reshape data by a factor of 256...
(312, 12576000)
Reshaped the output data to  
(49125, 312)
Saving 49125 samples to disk with filenamuber offset 147375...
Serialization complete. 49125 Files saved in 'data_slices/input'.


In [3]:

# from common import frame_size,reshape_to_nn_input



# nn_input=reshape_to_nn_input(filterbank_out)
# filterbank_out=None
# if use_augmentations:
#     nn_input_all=np.concatenate((nn_input,nn_input,nn_input,nn_input),axis=0)
#     nsamples=nn_input.shape[0];
#     nn_input_all[range(nsamples,2*nsamples)]=reshape_to_nn_input(filterbank_out_hi)
#     filterbank_out_hi=None
    
#     nn_input_all[range(2*nsamples,3*nsamples)]=reshape_to_nn_input(filterbank_out_mid)
#     filterbank_out_mid=None
    
#     nn_input_all[range(3*nsamples,4*nsamples)]=reshape_to_nn_input(filterbank_out_dist)
#     filterbank_out_dist=None
    
    
#     print(nn_input_all.shape)

In [4]:
# from common import save_data_slices
# output_dir_input = 'data_slices/input'
# if use_augmentations:
#     save_data_slices(output_dir_input,nn_input_all,batch_size)
# else:
#     save_data_slices(output_dir_input,nn_input,batch_size)

In [5]:
nn_input_all=None
nn_input=None